In [ ]:
# Unified Analysis Dashboard
## Interactive Analysis for Guardrailing, Content Quality, and Tool Calling Metrics

This notebook provides comprehensive analysis capabilities for evaluating model performance across three key dimensions:
- **Guardrailing Analysis**: Classification metrics for safe/unsafe content detection
- **Content Analysis**: Quality metrics including ROUGE, BERT, and semantic similarity
- **Tool Calling Analysis**: Metrics for tool selection, schema reliability, and variable handling

In [ ]:
import json
from pathlib import Path
from typing import Dict, Optional
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

## Data Reading Module
Load evaluation results from specified directories for interactive analysis

In [ ]:
# ============================================================================
# DATA LOADING AND UNIFIED ANALYSIS
# ============================================================================

def load_all_evaluation_data(eval_dir: str, include_timestamp: bool = False) -> pd.DataFrame:
    """Load and join all evaluation data into a single DataFrame"""
    if not eval_dir or not Path(eval_dir).exists():
        print(f"Error: {eval_dir} does not exist")
        return pd.DataFrame()

    all_data = []

    # Load data from each timestamp/model directory
    for timestamp_dir in Path(eval_dir).glob("*"):
        if not timestamp_dir.is_dir():
            continue

        for model_dir in timestamp_dir.glob("*"):
            if not model_dir.is_dir():
                continue

            model_name = f"{timestamp_dir.name}_{model_dir.name}" if include_timestamp else model_dir.name
            row = {"model": model_name, "timestamp": timestamp_dir.name}

            # Load tool calling data
            tool_file = model_dir / "tool_calling" / "per_item_results.parquet"
            if tool_file.exists():
                tool_df = pd.read_parquet(tool_file)
                for col in tool_df.columns:
                    if col.startswith("score_"):
                        metric = col.replace("score_", "")
                        row[f"tool_{metric}"] = tool_df[col].mean()
                row["tool_samples"] = len(tool_df)

            # Load content data
            content_file = model_dir / "content" / "per_item_results.parquet"
            if content_file.exists():
                content_df = pd.read_parquet(content_file)
                existing_cols = [col for col in ALL_METRIC_COLS if col in content_df.columns]
                if existing_cols:
                    content_df = content_df[content_df[existing_cols].notna().any(axis=1)]
                    for col in content_df.columns:
                        if col.startswith("score_"):
                            metric = col.replace("score_", "")
                            row[f"content_{metric}"] = content_df[col].mean()
                    row["content_samples"] = len(content_df)

            # Load guardrailing data
            guard_file = model_dir / "guardrailing" / "per_item_results.parquet"
            if guard_file.exists():
                guard_df = pd.read_parquet(guard_file)
                # Compute classification metrics
                y_true = guard_df["ground_truth_label"].values
                y_pred = guard_df["predicted_label"].values
                pos_label = "unsafe"

                row["guard_accuracy"] = float(accuracy_score(y_true, y_pred))
                row["guard_precision"] = float(precision_score(y_true, y_pred, pos_label=pos_label, zero_division=0))
                row["guard_recall"] = float(recall_score(y_true, y_pred, pos_label=pos_label, zero_division=0))
                row["guard_f1"] = float(f1_score(y_true, y_pred, pos_label=pos_label, zero_division=0))
                row["guard_samples"] = len(guard_df)

            if len(row) > 2:  # Has at least some data
                all_data.append(row)

    return pd.DataFrame(all_data) if all_data else pd.DataFrame()


# ============================================================================
# SQL QUERYING UTILITIES
# ============================================================================

try:
    import pandasql as psql
    HAS_SQL = True
except ImportError:
    HAS_SQL = False
    print("pandasql not available. Install with: pip install pandasql")

def sql_query(df: pd.DataFrame, query: str) -> pd.DataFrame:
    """Execute SQL query on DataFrame"""
    if not HAS_SQL:
        print("pandasql not available. Install with: pip install pandasql")
        return pd.DataFrame()

    try:
        return psql.sqldf(query, locals())
    except Exception as e:
        print(f"SQL Error: {e}")
        return pd.DataFrame()


def get_metric_ranges(df: pd.DataFrame) -> pd.DataFrame:
    """Get min/max ranges for all metrics"""
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    return df[numeric_cols].agg(['min', 'max', 'mean', 'std']).round(4)


def find_extreme_values(df: pd.DataFrame, metric: str, threshold: float, direction: str = "low") -> pd.DataFrame:
    """Find models with extreme metric values"""
    if metric not in df.columns:
        print(f"Metric {metric} not found in data")
        return pd.DataFrame()

    if direction == "low":
        return df[df[metric] < threshold].sort_values(metric)
    else:
        return df[df[metric] > threshold].sort_values(metric, ascending=False)


# ============================================================================
# EXAMPLE SQL QUERIES
# ============================================================================

EXAMPLE_QUERIES = {
    "low_accuracy_guardrailing": """
        SELECT model, guard_accuracy, guard_f1, guard_samples
        FROM df
        WHERE guard_accuracy < 0.8
        ORDER BY guard_accuracy ASC
    """,

    "high_tool_hallucination": """
        SELECT model, tool_tool_hallucination_rate, tool_tool_pick_up_rate, tool_samples
        FROM df
        WHERE tool_tool_hallucination_rate > 0.1
        ORDER BY tool_tool_hallucination_rate DESC
    """,

    "content_quality_issues": """
        SELECT model, content_rouge1, content_bert_f1, content_content_similarity, content_samples
        FROM df
        WHERE content_rouge1 < 0.5 OR content_bert_f1 < 0.6
        ORDER BY content_rouge1 ASC
    """,

    "best_overall_models": """
        SELECT model,
               (COALESCE(guard_accuracy, 0) + COALESCE(content_rouge1, 0) + COALESCE(tool_exact_match, 0)) / 3 as avg_score,
               guard_accuracy, content_rouge1, tool_exact_match
        FROM df
        WHERE guard_accuracy IS NOT NULL AND content_rouge1 IS NOT NULL AND tool_exact_match IS NOT NULL
        ORDER BY avg_score DESC
        LIMIT 5
    """,

    "missing_data_check": """
        SELECT model,
               CASE WHEN guard_accuracy IS NULL THEN 1 ELSE 0 END as missing_guard,
               CASE WHEN content_rouge1 IS NULL THEN 1 ELSE 0 END as missing_content,
               CASE WHEN tool_exact_match IS NULL THEN 1 ELSE 0 END as missing_tool
        FROM df
        WHERE guard_accuracy IS NULL OR content_rouge1 IS NULL OR tool_exact_match IS NULL
    """
}

In [ ]:
# Configuration
EVAL_DIR = ""  # SET THIS: Path to evaluation results directory
INCLUDE_TIMESTAMP = False  # Set to True if model dirs are organized by timestamp

# Load unified evaluation data
unified_df = pd.DataFrame()
if EVAL_DIR:
    unified_df = load_all_evaluation_data(EVAL_DIR, INCLUDE_TIMESTAMP)
    if not unified_df.empty:
        print(f"Loaded data for {len(unified_df)} models")
        print(f"Available metrics: {list(unified_df.columns)}")
        print("\nData preview:")
        print(unified_df.head())
    else:
        print("No data loaded. Check EVAL_DIR path.")
else:
    print("Configure EVAL_DIR above to load evaluation data")

## Analyser Classes
Base analyser and specialized analysers for different evaluation types

In [ ]:
class GuardrailingAnalyser(BaseAnalyser):
    """Analyser for guardrailing (safe/unsafe classification) metrics"""

    def compute_classification_metrics(self, df: pd.DataFrame) -> Dict:
        """Compute classification metrics from ground truth and predictions"""
        y_true = df["ground_truth_label"].values
        y_pred = df["predicted_label"].values
        pos_label = "unsafe"
        cm = confusion_matrix(y_true, y_pred, labels=["safe", pos_label])
        tn, fp, fn, tp = cm.ravel()

        return {
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, pos_label=pos_label, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, pos_label=pos_label, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, pos_label=pos_label, zero_division=0)),
            "total_samples": len(df),
            "true_positives": int(tp),
            "false_positives": int(fp),
            "true_negatives": int(tn),
            "false_negatives": int(fn),
        }

    def compute_summary(self, df: pd.DataFrame) -> Dict:
        """Override to include classification metrics and distributions"""
        return {
            "num_samples": len(df),
            "metrics": self.compute_classification_metrics(df),
            "ground_truth_distribution": df["ground_truth_label"].value_counts().to_dict(),
            "predicted_distribution": df["predicted_label"].value_counts().to_dict(),
        }

## Content Analyser
Quality metrics including ROUGE, BERT, semantic similarity, and conversational quality

In [ ]:
# Content Analysis Metrics
CONTENT_METRICS = [
    "levenshtein_ratio", "rouge1", "rougeL", "bert_precision", "bert_recall",
    "bert_f1", "content_similarity", "context_accuracy", "content_politeness",
    "content_helpfulness",
]
LOWER_IS_BETTER_CONTENT = ["levenshtein_distance"]

ALL_METRIC_COLS = [f"score_{m}" for m in CONTENT_METRICS + LOWER_IS_BETTER_CONTENT]


class ContentAnalyser(BaseAnalyser):
    """Analyser for content quality metrics (ROUGE, BERT, semantic similarity, etc)"""
    pass  # Uses base class methods

## Guardrailing Analyser
Classification metrics for safe/unsafe content detection

In [ ]:
# Tool Calling Metrics
POSITIVE_METRICS = [
    "when2call", "schema_reliability_raw", "schema_reliability_processed",
    "tool_pick_up_rate", "variable_pickup_rate", "variable_correct_rate", "exact_match",
]
NEGATIVE_METRICS = [
    "tool_hallucination_rate", "tool_additional_rate",
    "variable_hallucination_rate", "variable_additional_rate",
]


class ToolCallingAnalyser(BaseAnalyser):
    """Analyser for tool calling metrics (schema reliability, variable handling, etc)"""
    pass  # Uses base class methods

## Unified Data Analysis with SQL Queries
Analyze the combined evaluation data using SQL queries

In [ ]:
# ============================================================================
# UNIFIED DATA ANALYSIS
# ============================================================================

if not unified_df.empty:
    print("=== UNIFIED EVALUATION DATA ===")
    print(f"Shape: {unified_df.shape}")
    print(f"Models: {len(unified_df)}")
    print(f"Metrics: {len(unified_df.columns)}")
    print("\nColumns:", list(unified_df.columns))

    # Show metric ranges
    print("\n=== METRIC RANGES ===")
    ranges = get_metric_ranges(unified_df)
    print(ranges)

    # Example SQL queries
    print("\n=== EXAMPLE SQL QUERIES ===")

    # 1. Models with low guardrailing accuracy
    print("\n1. Models with low guardrailing accuracy (< 0.8):")
    result = sql_query(unified_df, EXAMPLE_QUERIES["low_accuracy_guardrailing"])
    if not result.empty:
        print(result)
    else:
        print("No results or pandasql not available")

    # 2. Models with high tool hallucination rates
    print("\n2. Models with high tool hallucination rates (> 0.1):")
    result = sql_query(unified_df, EXAMPLE_QUERIES["high_tool_hallucination"])
    if not result.empty:
        print(result)

    # 3. Models with content quality issues
    print("\n3. Models with content quality issues:")
    result = sql_query(unified_df, EXAMPLE_QUERIES["content_quality_issues"])
    if not result.empty:
        print(result)

    # 4. Best overall models
    print("\n4. Best overall models (average of key metrics):")
    result = sql_query(unified_df, EXAMPLE_QUERIES["best_overall_models"])
    if not result.empty:
        print(result)

    # 5. Missing data check
    print("\n5. Models with missing data:")
    result = sql_query(unified_df, EXAMPLE_QUERIES["missing_data_check"])
    if not result.empty:
        print(result)

else:
    print("No unified data available. Configure EVAL_DIR and run data loading cell.")

## Custom SQL Queries
Write your own SQL queries to analyze the unified evaluation data

In [ ]:
# ============================================================================
# CUSTOM SQL QUERIES
# ============================================================================

# Example: Write your own SQL queries here
# Available columns: model, timestamp, tool_*, content_*, guard_*, *_samples

CUSTOM_QUERY = """
SELECT model,
       tool_exact_match,
       content_rouge1,
       guard_accuracy,
       tool_samples + content_samples + guard_samples as total_samples
FROM df
WHERE tool_exact_match > 0.8 AND content_rouge1 > 0.7 AND guard_accuracy > 0.85
ORDER BY tool_exact_match DESC
"""

# Execute custom query
if not unified_df.empty and HAS_SQL:
    print("=== CUSTOM QUERY RESULTS ===")
    result = sql_query(unified_df, CUSTOM_QUERY)
    if not result.empty:
        print(result)
    else:
        print("No results found")
else:
    print("Unified data not available or pandasql not installed")

# ============================================================================
# QUICK ANALYSIS FUNCTIONS
# ============================================================================

def analyze_metric(df: pd.DataFrame, metric: str, threshold_low: float = None, threshold_high: float = None):
    """Quick analysis of a specific metric"""
    if metric not in df.columns:
        print(f"Metric {metric} not found")
        return

    print(f"\n=== ANALYSIS: {metric} ===")
    print(f"Available data points: {df[metric].notna().sum()}")
    print(f"Mean: {df[metric].mean():.4f}")
    print(f"Std: {df[metric].std():.4f}")
    print(f"Min: {df[metric].min():.4f}")
    print(f"Max: {df[metric].max():.4f}")

    if threshold_low is not None:
        low_count = (df[metric] < threshold_low).sum()
        print(f"Below {threshold_low}: {low_count} models")

    if threshold_high is not None:
        high_count = (df[metric] > threshold_high).sum()
        print(f"Above {threshold_high}: {high_count} models")

# Example usage:
# analyze_metric(unified_df, "guard_accuracy", threshold_low=0.8)
# analyze_metric(unified_df, "tool_exact_match", threshold_high=0.9)